# AgentAlign Lab — Eval-Only (SFT + DPO on Test Split)

The base model eval is already done (263 trajectories, 8.0% pass rate).
This notebook runs **only** the SFT and DPO adapter evaluations on the same test tasks,
then computes bootstrap CIs and a failure taxonomy.

**Prerequisites**: Upload a dataset zip containing:
- The full repo (from `outputs/gpu_handoff.zip`)
- The trained adapters (from the previous `agentalign_results.zip`)

In [ ]:
# 0. Check GPU
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 1. Setup — clone repo, install, copy adapters
!git clone https://github.com/hey-shiv/AgentAlign-Lab.git /kaggle/working/repo 2>/dev/null || true
%cd /kaggle/working/repo

# Regenerate tasks (deterministic, same seed)
!pip install -e '.[ml,dashboard]' -q
!python scripts/01_generate_tasks.py

# Copy adapters from the dataset input
import shutil, os
from pathlib import Path

# Try to find the adapters in /kaggle/input
input_root = Path('/kaggle/input')
for d in input_root.iterdir():
    print(f'  Dataset: {d.name}')
    # Look for adapter dirs
    for adapter_name in ['qwen_dpo_final', 'qwen_sft_final']:
        candidates = list(d.rglob(adapter_name))
        if candidates:
            src = candidates[0]
            dst = Path(f'outputs/adapters/{adapter_name}')
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'    ✓ Copied {adapter_name} from {src}')
        else:
            print(f'    ✗ {adapter_name} not found in {d.name}')

# Also copy existing base test results if available
for d in input_root.iterdir():
    for subdir in ['runs/llm_test', 'data/trajectories/scored_llm_test']:
        candidates = list(d.rglob(subdir.split('/')[-1]))
        if candidates:
            src = candidates[0]
            dst = Path(subdir)
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'    ✓ Copied {subdir} from {src}')

# Verify
for p in ['outputs/adapters/qwen_dpo_final/adapter_config.json',
          'outputs/adapters/qwen_sft_final/adapter_config.json',
          'data/tasks/test.jsonl']:
    print(f'  {"✓" if Path(p).exists() else "✗"} {p}')

In [ ]:
# 2. Verify task splits
import json, collections
for split in ['train', 'val', 'test']:
    tasks = [json.loads(l) for l in open(f'data/tasks/{split}.jsonl')]
    fam = collections.Counter(t['family'] for t in tasks)
    print(f'{split}: {len(tasks)} tasks')
    for k,v in sorted(fam.items()): print(f'  {k}: {v}')
    print()

In [ ]:
# 3. Run SFT adapter eval on test split (4 reps, same seeds as base)
!python scripts/02_run_agent.py \
    --split test --agent qwen_sft \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter outputs/adapters/qwen_sft_final \
    --out runs/llm_test \
    --repetitions 4 \
    --temperature 0.8 --seed 42 --max-steps 8 \
    --device cuda --resume

In [ ]:
# 4. Run DPO adapter eval on test split (4 reps, same seeds as base)
!python scripts/02_run_agent.py \
    --split test --agent qwen_dpo \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter outputs/adapters/qwen_dpo_final \
    --out runs/llm_test \
    --repetitions 4 \
    --temperature 0.8 --seed 42 --max-steps 8 \
    --device cuda --resume

In [ ]:
# 5. Score all test trajectories
!python scripts/03_score_trajectories.py \
    --runs-dir runs/llm_test \
    --out-dir data/trajectories/scored_llm_test \
    --clear

In [ ]:
# 6. Compute metrics and bootstrap CIs
import json, numpy as np
from pathlib import Path
from collections import defaultdict

scored_dir = Path('data/trajectories/scored_llm_test')
trajectories = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if line.strip():
            trajectories.append(json.loads(line))

print(f'Total scored test trajectories: {len(trajectories)}')

# Group by agent
by_agent = defaultdict(list)
for t in trajectories:
    by_agent[t.get('agent_id', 'unknown')].append(t)

print(f'Agents found: {list(by_agent.keys())}')
for a, trajs in sorted(by_agent.items()):
    print(f'  {a}: {len(trajs)} trajectories')

import re

def get_family(task_id):
    return re.sub(r'_\d+$', '', task_id) if task_id else 'unknown'

# Compute metrics per agent
def compute_agent_metrics(trajs):
    total = len(trajs)
    passed = sum(1 for t in trajs if t.get('verifier_result', {}).get('passed', False))
    unsafe = sum(t.get('verifier_result', {}).get('unsafe_actions', 0) for t in trajs)
    failed_cmds = sum(t.get('verifier_result', {}).get('failed_commands', 0) for t in trajs)
    invalid = sum(t.get('verifier_result', {}).get('invalid_actions', 0) for t in trajs)
    avg_steps = np.mean([len(t.get('steps', [])) for t in trajs])
    return {
        'total': total,
        'passed': passed,
        'pass_rate': round(passed / max(total, 1), 4),
        'unsafe_actions': int(unsafe),
        'unsafe_rate': round(unsafe / max(total, 1), 4),
        'failed_commands': int(failed_cmds),
        'invalid_actions': int(invalid),
        'avg_steps': round(float(avg_steps), 2),
    }

results = {}
for agent, trajs in sorted(by_agent.items()):
    m = compute_agent_metrics(trajs)
    results[agent] = m
    print(f'\n{agent}: {m["passed"]}/{m["total"]} pass ({m["pass_rate"]*100:.1f}%), '
          f'unsafe={m["unsafe_actions"]}, invalid={m["invalid_actions"]}, '
          f'avg_steps={m["avg_steps"]}')

# Per-family breakdown
print('\n--- Per-family pass rates ---')
for agent in sorted(by_agent):
    trajs = by_agent[agent]
    family_stats = defaultdict(lambda: {'total': 0, 'passed': 0, 'unsafe': 0})
    for t in trajs:
        family = get_family(t.get('task_id', ''))
        family_stats[family]['total'] += 1
        if t.get('verifier_result', {}).get('passed', False):
            family_stats[family]['passed'] += 1
        family_stats[family]['unsafe'] += t.get('verifier_result', {}).get('unsafe_actions', 0)
    print(f'\n{agent}:')
    for fam in sorted(family_stats):
        s = family_stats[fam]
        rate = s['passed'] / max(s['total'], 1) * 100
        print(f'  {fam}: {s["passed"]}/{s["total"]} ({rate:.1f}%) unsafe={s["unsafe"]}')
    results[agent]['per_family'] = {k: dict(v) for k, v in family_stats.items()}

# Bootstrap CI for pass-rate differences
def paired_bootstrap_ci(agent_a_trajs, agent_b_trajs, n_boot=10000, seed=42):
    rng = np.random.RandomState(seed)
    a_by_task = defaultdict(list)
    b_by_task = defaultdict(list)
    for t in agent_a_trajs:
        a_by_task[t['task_id']].append(1.0 if t.get('verifier_result', {}).get('passed', False) else 0.0)
    for t in agent_b_trajs:
        b_by_task[t['task_id']].append(1.0 if t.get('verifier_result', {}).get('passed', False) else 0.0)
    common_tasks = sorted(set(a_by_task) & set(b_by_task))
    if not common_tasks:
        return 0, 0, 0, 1.0
    a_rates = np.array([np.mean(a_by_task[t]) for t in common_tasks])
    b_rates = np.array([np.mean(b_by_task[t]) for t in common_tasks])
    diffs = b_rates - a_rates
    observed_diff = np.mean(diffs)
    boot_diffs = []
    n = len(diffs)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        boot_diffs.append(np.mean(diffs[idx]))
    boot_diffs = np.array(boot_diffs)
    ci_lo = float(np.percentile(boot_diffs, 2.5))
    ci_hi = float(np.percentile(boot_diffs, 97.5))
    p_value = float(np.mean(np.abs(boot_diffs - np.mean(boot_diffs)) >= abs(observed_diff)))
    return round(observed_diff, 4), round(ci_lo, 4), round(ci_hi, 4), round(p_value, 4)

print('\n--- Bootstrap 95% CIs ---')
comparisons = [
    ('dpo_vs_base', 'qwen_base', 'qwen_dpo', 'DPO - Base'),
    ('sft_vs_base', 'qwen_base', 'qwen_sft', 'SFT - Base'),
    ('dpo_vs_sft', 'qwen_sft', 'qwen_dpo', 'DPO - SFT'),
]
for key, a, b, label in comparisons:
    if a in by_agent and b in by_agent:
        diff, ci_lo, ci_hi, p = paired_bootstrap_ci(by_agent[a], by_agent[b])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        spans_zero = 'YES (no detectable improvement)' if ci_lo <= 0 <= ci_hi else 'NO'
        print(f'{label}: {diff:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}] p={p} {sig}')
        print(f'  CI spans zero: {spans_zero}')
        results[key] = {'diff': diff, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'p_value': p}
    else:
        print(f'{label}: SKIPPED (agent not found)')

Path('outputs/evals').mkdir(parents=True, exist_ok=True)
Path('outputs/evals/eval_results.json').write_text(json.dumps(results, indent=2))
print('\n✅ Saved to outputs/evals/eval_results.json')

In [ ]:
# 7. Failure taxonomy — 10 DPO test failures
import json, re
from pathlib import Path

scored_dir = Path('data/trajectories/scored_llm_test')
failures = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        t = json.loads(line)
        if t.get('agent_id') == 'qwen_dpo' and not t.get('verifier_result', {}).get('passed', False):
            failures.append(t)

print(f'Total DPO failures on test: {len(failures)}')

# Classify failures
def classify_failure(t):
    vr = t.get('verifier_result', {})
    steps = t.get('steps', [])
    n_steps = len(steps)
    unsafe = vr.get('unsafe_actions', 0)
    invalid = vr.get('invalid_actions', 0)
    failed = vr.get('failed_commands', 0)
    
    if unsafe > 0:
        return 'unsafe_action'
    if invalid > n_steps * 0.5 and n_steps > 0:
        return 'mostly_invalid_actions'
    if n_steps >= 7:
        return 'max_steps_exhausted'
    if failed > 0 and n_steps <= 3:
        return 'early_command_failure'
    if n_steps <= 2:
        return 'premature_termination'
    return 'wrong_output'

sample = failures[:10]
taxonomy = []
for i, f in enumerate(sample):
    vr = f.get('verifier_result', {})
    family = re.sub(r'_\d+$', '', f.get('task_id', ''))
    label = classify_failure(f)
    n_steps = len(f.get('steps', []))
    entry = {
        'task_id': f.get('task_id'),
        'family': family,
        'failure_category': label,
        'n_steps': n_steps,
        'unsafe_actions': vr.get('unsafe_actions', 0),
        'invalid_actions': vr.get('invalid_actions', 0),
        'failed_commands': vr.get('failed_commands', 0),
        'score': vr.get('score', 0),
    }
    taxonomy.append(entry)
    print(f'{i+1}. {entry["task_id"]}: {label} (family={family}, steps={n_steps}, '
          f'unsafe={entry["unsafe_actions"]}, invalid={entry["invalid_actions"]})')

# Summary
from collections import Counter
all_labels = [classify_failure(f) for f in failures]
print(f'\nFailure distribution (all {len(failures)} DPO failures):')
for label, count in Counter(all_labels).most_common():
    print(f'  {label}: {count} ({count/len(failures)*100:.0f}%)')

taxonomy_data = {
    'sample_failures': taxonomy,
    'total_failures': len(failures),
    'failure_distribution': dict(Counter(all_labels).most_common()),
}
Path('outputs/evals/failure_taxonomy.json').write_text(json.dumps(taxonomy_data, indent=2))
print('\n✅ Saved to outputs/evals/failure_taxonomy.json')

In [ ]:
# 8. Package results for download
import shutil
from pathlib import Path

export_dir = Path('/kaggle/working/export')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(exist_ok=True)

for src in [
    'outputs/evals',
    'data/trajectories/scored_llm_test',
    'runs/llm_test',
]:
    src_path = Path(src)
    if not src_path.exists():
        print(f'  SKIP: {src}')
        continue
    dst = export_dir / src
    if src_path.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dst)
    print(f'  ✓ {src}')

shutil.make_archive('/kaggle/working/agentalign_eval_results', 'zip', export_dir)
print(f'\n✅ Download: /kaggle/working/agentalign_eval_results.zip')
print(f'Then run locally:')
print(f'  python scripts/10_import_colab_artifacts.py --zip-path agentalign_eval_results.zip --force')